# Re-engagement CRM clustering approaches

Exploration notebook for `data/raw/clustering.zip`.

Goal: compare three different clustering approaches for CRM/CDP-style guest segmentation. The model path should stay deterministic: input features -> rule segment or learned cluster id. Cluster naming here is static and heuristic, not LLM-generated.

Run summary:

- Source rows: **519,874**
- Learned-cluster candidates: **304,616**
- Fit rows used in this full local run: **304,616**
- Rule outputs outside learned clustering: **COLD_CONTACTS** and **NOT_TARGETABLE**
- Runtime on local Mac: about **54 seconds** for all three approaches after data load

Rule segment counts:

| Segment | Rows | Meaning |
|---|---:|---|
| `LEARNED_CLUSTER` | 304,616 | Reachable contacts with send history; used for learned clustering |
| `COLD_CONTACTS` | 157,639 | Reachable, but never sent / cold-start contacts |
| `NOT_TARGETABLE` | 57,619 | Not reachable, opted out, excluded, no email, or unknown reachability |


In [ ]:
from pathlib import Path
import json
import pandas as pd

REPORT_PATH = Path('../data/processed/clustering_experiments/approach_comparison.json')
if not REPORT_PATH.exists():
    REPORT_PATH = Path('data/processed/clustering_experiments/approach_comparison.json')
report = json.loads(REPORT_PATH.read_text())

pd.DataFrame([
    {'segment': k, 'rows': v}
    for k, v in report['rule_segment_counts'].items()
]).sort_values('rows', ascending=False)

## Approach 1: CDP mixed k-means, k=10

Centroid clustering over engagement, booking, profile, and normalized categorical features. This is now the recommended first service candidate.

| Cluster | Static name | What it represents | Rows | Profile | Reality-check guest examples |
|---:|---|---|---:|---|---|
| 0 | Tichí příjemci | Nízkoreakční příjemci bez spolehlivého open/click signálu. | 64,516 | eng=no_open, open90=0.083, click90=0.000, sents90=0.71, last_open_med=332, stays=0.00, spend_rank=0.889 | `57df3b5c81f5`: no_open, s90=1, o90=0, c90=0, stays=0, spend_rank=1.000<br>`a51145029acf`: no_open, s90=1, o90=0, c90=0, stays=0, spend_rank=1.000 |
| 1 | Středně aktivní CRM kontakty | Střední segment bez výrazného booking, click nebo recency signálu. | 19,175 | eng=fading, open90=0.152, click90=0.002, sents90=3.05, last_open_med=164, stays=1.34, spend_rank=0.791 | `35fd2ea6421e`: no_open, s90=3, o90=0, c90=0, stays=1, spend_rank=0.743<br>`503cb85827e1`: no_open, s90=3, o90=0, c90=0, stays=1, spend_rank=0.743 |
| 2 | Oslovení bez reakce | Kontakty, které dostávají e-maily, ale v posledních oknech na ně prakticky nereagují. | 26,450 | eng=no_open, open90=0.060, click90=0.003, sents90=4.51, last_open_med=66, stays=0.00, spend_rank=0.939 | `59b4026dda42`: no_open, s90=2, o90=0, c90=0, stays=0, spend_rank=1.000<br>`0f049feae06c`: no_open, s90=2, o90=0, c90=0, stays=0, spend_rank=1.000 |
| 3 | Častější hosté bez aktuální reakce | Bookingově silnější hosté, ale bez aktuální e-mailové odezvy. | 18,088 | eng=no_open, open90=0.444, click90=0.000, sents90=0.68, last_open_med=132, stays=3.33, spend_rank=0.887 | `7cb42f5fde4c`: no_open, s90=1, o90=0, c90=0, stays=1, spend_rank=0.976<br>`e749bf6686aa`: no_open, s90=1, o90=0, c90=0, stays=1, spend_rank=0.982 |
| 4 | Aktivní čtenáři | Aktivní kontakty, které nedávno otevíraly e-maily, ale méně klikají. | 21,133 | eng=healthy, open90=0.803, click90=0.045, sents90=3.57, last_open_med=9, stays=0.55, spend_rank=0.884 | `2bf2b3ad5c5f`: healthy, s90=2, o90=2, c90=0, stays=0, spend_rank=1.000<br>`63e95eb809cc`: healthy, s90=2, o90=2, c90=0, stays=0, spend_rank=1.000 |
| 5 | Dřívější hosté s vyhasínající odezvou | Hosté s dřívější odezvou, která je dnes už slabá a starší. | 30,790 | eng=fading, open90=0.362, click90=0.000, sents90=0.09, last_open_med=410, stays=1.19, spend_rank=0.747 | `26b5218c2b5a`: no_open, s90=0, o90=0, c90=0, stays=1, spend_rank=0.796<br>`9ad10c557a7e`: no_open, s90=0, o90=0, c90=0, stays=1, spend_rank=0.789 |
| 6 | Kontakty bez pobytové historie | Kontakty s nulovou nebo minimální pobytovou historií a nízkou nebo starou e-mailovou aktivitou. | 56,581 | eng=fading, open90=0.515, click90=0.007, sents90=0.29, last_open_med=262, stays=0.01, spend_rank=0.926 | `1605ea17a986`: no_open, s90=0, o90=0, c90=0, stays=0, spend_rank=1.000<br>`ccb2a856d578`: no_open, s90=0, o90=0, c90=0, stays=0, spend_rank=1.000 |
| 7 | Dřívější hosté s velmi nízkou/žádnou odezvou | Dřívější hosté s booking historií, u kterých je e-mailová odezva velmi nízká nebo žádná. | 39,792 | eng=fading, open90=0.258, click90=0.000, sents90=0.19, last_open_med=708, stays=1.23, spend_rank=0.730 | `5e0959308d18`: no_open, s90=0, o90=0, c90=0, stays=1, spend_rank=0.797<br>`5b7e6a677438`: no_open, s90=0, o90=0, c90=0, stays=1, spend_rank=0.790 |
| 8 | Aktivní klikající hosté | Hosté s pobyty a vysokou recent open/click aktivitou. | 10,568 | eng=healthy, open90=0.849, click90=0.711, sents90=2.44, last_open_med=39, stays=2.00, spend_rank=0.867 | `49bcadce7f38`: healthy, s90=1, o90=1, c90=1, stays=1, spend_rank=0.957<br>`398687f8786a`: healthy, s90=1, o90=1, c90=1, stays=1, spend_rank=0.922 |
| 9 | Hodnotnější dřívější hosté bez odezvy | Vyšší booking/spend rank, ale slabá nebo žádná současná odezva. | 17,523 | eng=fading, open90=0.168, click90=0.007, sents90=0.78, last_open_med=563, stays=1.74, spend_rank=0.840 | `357b4e71f421`: no_open, s90=0, o90=0, c90=0, stays=1, spend_rank=0.858<br>`55c2679156aa`: no_open, s90=0, o90=0, c90=0, stays=1, spend_rank=0.858 |

Read: k=10 is more useful for CDP because it separates active readers, active clickers, no-open recipients, no-stay contacts, and multiple tiers of previous guests.

## Archived k=5 top and edge guest examples

This section is kept as the original k=5 baseline. The current recommended CDP model is k=10; use the dynamic k=10 example table below for current examples.

`distance_to_cluster_profile` is a normalized distance from the cluster median profile. Lower means the guest is more typical for the cluster; higher means the guest sits near the edge of the cluster.

### Cluster 0 - Oslovení bez reakce
Kontakty, které dostávají e-maily, ale v posledních oknech na ně prakticky nereagují.

**3 nejtypičtější kontakty**

| guest_ref | distance | engagement | s90 | o90 | c90 | open90 | click90 | days_open | days_click | stays | spend_rank | stay_recency_rank |
|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| `06b11ce14ae9` | 0.200 | no_open | 2 | 0 | 0 | 0.000 | 0.000 | N/A | N/A | 0 | 1.000 | 1.000 |
| `d172cddf79af` | 0.200 | no_open | 2 | 0 | 0 | 0.000 | 0.000 | N/A | N/A | 0 | 1.000 | 1.000 |
| `eba0c2d1fa9f` | 0.200 | no_open | 2 | 0 | 0 | 0.000 | 0.000 | N/A | N/A | 0 | 1.000 | 1.000 |

**3 okrajové kontakty v clusteru**

| guest_ref | distance | engagement | s90 | o90 | c90 | open90 | click90 | days_open | days_click | stays | spend_rank | stay_recency_rank |
|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| `44a2f6584fe5` | 56.811 | healthy | 1 | 1 | 0 | 1.000 | 0.000 | 88 | N/A | 0 | 0.284 | 0.235 |
| `b23729ea2d64` | 44.292 | healthy | 1 | 1 | 0 | 1.000 | 0.000 | 89 | N/A | 0 | 0.432 | 0.437 |
| `86375f7e2d90` | 44.292 | healthy | 1 | 1 | 0 | 1.000 | 0.000 | 89 | N/A | 0 | 0.432 | 0.437 |

### Cluster 1 - Dřívější hosté s velmi nízkou/žádnou odezvou
Dřívější hosté s booking historií, u kterých je e-mailová odezva velmi nízká nebo žádná.

**3 nejtypičtější kontakty**

| guest_ref | distance | engagement | s90 | o90 | c90 | open90 | click90 | days_open | days_click | stays | spend_rank | stay_recency_rank |
|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| `8a9e9d74f98e` | 0.011 | no_open | 0 | 0 | 0 | N/A | N/A | N/A | N/A | 1 | 0.858 | 0.571 |
| `5307930734eb` | 0.032 | no_open | 0 | 0 | 0 | N/A | N/A | N/A | N/A | 1 | 0.858 | 0.578 |
| `b925b1b2b345` | 0.032 | no_open | 0 | 0 | 0 | N/A | N/A | N/A | N/A | 1 | 0.858 | 0.578 |

**3 okrajové kontakty v clusteru**

| guest_ref | distance | engagement | s90 | o90 | c90 | open90 | click90 | days_open | days_click | stays | spend_rank | stay_recency_rank |
|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| `1b4519b6a76a` | 145.243 | no_open | 0 | 0 | 0 | N/A | N/A | N/A | N/A | 144 | 0.416 | 0.348 |
| `94ef52ad9d10` | 73.728 | no_open | 1 | 0 | 0 | 0.000 | 0.000 | N/A | N/A | 73 | 0.995 | 0.646 |
| `fa9b7a9ed738` | 67.807 | fading | 0 | 0 | 0 | N/A | N/A | 450 | N/A | 66 | 0.416 | 0.292 |

### Cluster 2 - Aktivní klikači
Aktivní kontakty s čerstvými openy a nadprůměrnou klikací aktivitou.

**3 nejtypičtější kontakty**

| guest_ref | distance | engagement | s90 | o90 | c90 | open90 | click90 | days_open | days_click | stays | spend_rank | stay_recency_rank |
|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| `d53dc77ad69c` | 0.923 | healthy | 3 | 3 | 0 | 1.000 | 0.000 | 16 | N/A | 0 | 0.977 | 0.967 |
| `2c8a07d9741a` | 0.923 | healthy | 3 | 3 | 0 | 1.000 | 0.000 | 16 | N/A | 0 | 0.977 | 0.967 |
| `383be7453721` | 0.930 | healthy | 2 | 2 | 0 | 1.000 | 0.000 | 16 | N/A | 0 | 1.000 | 1.000 |

**3 okrajové kontakty v clusteru**

| guest_ref | distance | engagement | s90 | o90 | c90 | open90 | click90 | days_open | days_click | stays | spend_rank | stay_recency_rank |
|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| `1f341d1b4aad` | 2345.884 | healthy | 112 | 112 | 47 | 1.000 | 0.420 | 0 | 0 | 2205 | 1.000 | 1.000 |
| `68e80e89acf0` | 314.528 | healthy | 5 | 5 | 307 | 1.000 | 1.000 | 17 | 13 | 0 | 0.588 | 0.486 |
| `d68e1e652d11` | 262.605 | healthy | 6 | 5 | 8 | 0.833 | 1.000 | 18 | 20 | 248 | 0.996 | 0.975 |

### Cluster 3 - Dřívější hosté s klesající odezvou
Hosté s pobytovou historií a dřívější reakcí, kterým engagement postupně slábne.

**3 nejtypičtější kontakty**

| guest_ref | distance | engagement | s90 | o90 | c90 | open90 | click90 | days_open | days_click | stays | spend_rank | stay_recency_rank |
|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| `f671e8b64611` | 0.000 | fading | 0 | 0 | 0 | N/A | N/A | 254 | N/A | 1 | 0.891 | 0.787 |
| `7d5fde2844cb` | 0.002 | fading | 0 | 0 | 0 | N/A | N/A | 254 | N/A | 1 | 0.891 | 0.786 |
| `dcc2b3149974` | 0.002 | fading | 0 | 0 | 0 | N/A | N/A | 254 | N/A | 1 | 0.891 | 0.786 |

**3 okrajové kontakty v clusteru**

| guest_ref | distance | engagement | s90 | o90 | c90 | open90 | click90 | days_open | days_click | stays | spend_rank | stay_recency_rank |
|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| `6e0c52edf107` | 2521.860 | fading | 0 | 0 | 0 | N/A | N/A | 542 | 542 | 2521 | 1.000 | 0.939 |
| `912de4047cd0` | 920.589 | fading | 28 | 0 | 0 | 0.000 | 0.000 | 704 | N/A | 891 | 1.000 | 1.000 |
| `9bf3d9fa0b80` | 903.574 | fading | 0 | 0 | 0 | N/A | N/A | N/A | N/A | 904 | 1.000 | 0.741 |

### Cluster 4 - Kontakty bez pobytové historie
Kontakty s nulovou nebo minimální pobytovou historií a nízkou nebo starou e-mailovou aktivitou.

**3 nejtypičtější kontakty**

| guest_ref | distance | engagement | s90 | o90 | c90 | open90 | click90 | days_open | days_click | stays | spend_rank | stay_recency_rank |
|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| `d72126cb3b03` | 0.008 | no_open | 0 | 0 | 0 | N/A | N/A | N/A | 1068 | 0 | 0.999 | 0.885 |
| `5e10489975de` | 0.008 | no_open | 0 | 0 | 0 | N/A | N/A | N/A | N/A | 0 | 0.999 | 0.885 |
| `655996507c9e` | 0.008 | no_open | 0 | 0 | 0 | N/A | N/A | N/A | N/A | 0 | 0.999 | 0.885 |

**3 okrajové kontakty v clusteru**

| guest_ref | distance | engagement | s90 | o90 | c90 | open90 | click90 | days_open | days_click | stays | spend_rank | stay_recency_rank |
|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| `65364ff534ce` | 10.881 | healthy | 4 | 1 | 1 | 0.250 | 0.250 | 39 | 39 | 0 | 0.588 | 0.486 |
| `23ba47401ae0` | 10.512 | healthy | 2 | 2 | 0 | 1.000 | 0.000 | 36 | N/A | 0 | 0.460 | 0.385 |
| `b40122efcc1b` | 10.510 | healthy | 2 | 2 | 0 | 1.000 | 0.000 | 37 | N/A | 0 | 0.460 | 0.385 |


In [ ]:
# Current k=10 CDP k-means: 3 typical and 3 edge contacts per cluster.
km = next(approach for approach in report['approaches'] if approach['name'] == 'cdp_kmeans')
example_rows = []
for cluster in km['clusters']:
    for bucket, rows in [('typical', cluster['top_examples']), ('edge', cluster['edge_examples'])]:
        for example in rows:
            example_rows.append({
                'cluster_id': cluster['cluster_id'],
                'cluster_name': cluster['label'],
                'example_type': bucket,
                'guest_ref': example['guest_ref'],
                'distance': example['distance_to_cluster_profile'],
                'engagement_state': example['engagement_state'],
                'sents_90d': example['sents_90d'],
                'opens_90d': example['opens_90d'],
                'clicks_90d': example['clicks_90d'],
                'open_rate_90d': example['open_rate_90d'],
                'click_rate_90d': example['click_rate_90d'],
                'days_since_last_open': example['days_since_last_open'],
                'days_since_last_click': example['days_since_last_click'],
                'stays_total': example['stays_total'],
                'rank_spend_total': example['rank_spend_total'],
                'rank_recency_stay': example['rank_recency_stay'],
            })

pd.DataFrame(example_rows).sort_values(['cluster_id', 'example_type', 'distance'])

## Approach 2: RFM BIRCH

Incremental numeric clustering focused on recency, frequency, monetary rank, and engagement rank. This intentionally ignores most categorical profile fields.

| Cluster | Static name | Rows | Profile | Reality-check guest examples |
|---:|---|---:|---|---|
| 0 | Aktivní čtenáři | 39,579 | eng=healthy, open90=0.861, click90=0.000, sents90=2.25, last_open_med=29, stays=0.97, spend_rank=0.869 | `56ae7318f89b`: healthy, s90=1, o90=1, c90=0, stays=0, spend_rank=0.999<br>`387efc27b699`: healthy, s90=1, o90=1, c90=0, stays=0, spend_rank=1.000 |
| 1 | Tichí příjemci | 244,524 | eng=fading, open90=0.014, click90=0.000, sents90=0.90, last_open_med=417, stays=0.71, spend_rank=0.853 | `e5de25ac1e43`: no_open, s90=0, o90=0, c90=0, stays=0, spend_rank=1.000<br>`7de3454db0e2`: no_open, s90=0, o90=0, c90=0, stays=0, spend_rank=1.000 |
| 2 | Aktivní klikači | 157 | eng=healthy, open90=0.583, click90=0.193, sents90=2.62, last_open_med=3, stays=21.90, spend_rank=0.841 | `dda636f13f1e`: healthy, s90=1, o90=1, c90=0, stays=1, spend_rank=0.860<br>`9c3999b792a6`: healthy, s90=1, o90=1, c90=0, stays=1, spend_rank=0.819 |
| 3 | Aktivní klikači | 11,096 | eng=healthy, open90=0.852, click90=0.738, sents90=2.61, last_open_med=30, stays=1.32, spend_rank=0.877 | `383190e549bd`: healthy, s90=1, o90=1, c90=1, stays=1, spend_rank=0.976<br>`bb8111a7d072`: healthy, s90=1, o90=1, c90=1, stays=1, spend_rank=0.996 |
| 4 | Klikající zájemci | 9,260 | eng=healthy, open90=0.423, click90=0.054, sents90=4.38, last_open_med=42, stays=0.59, spend_rank=0.835 | `cda48dd196e1`: healthy, s90=3, o90=1, c90=0, stays=0, spend_rank=0.977<br>`77de47047d98`: healthy, s90=3, o90=1, c90=0, stays=0, spend_rank=0.977 |

Read: good for finding sharp behavioral outliers, but too imbalanced for the main CDP taxonomy.

## Approach 3: Latent SVD Gaussian mixture

Probabilistic clustering on a dense latent representation of mixed CRM features. This approach compresses numeric plus categorical signals before fitting a Gaussian mixture.

| Cluster | Static name | Rows | Profile | Reality-check guest examples |
|---:|---|---:|---|---|
| 0 | Kontakty bez pobytové historie | 88,218 | eng=fading, open90=0.008, click90=0.000, sents90=0.54, last_open_med=354, stays=0.02, spend_rank=0.992 | `55b55ecd49b6`: no_open, s90=0, o90=0, c90=0, stays=0, spend_rank=1.000<br>`078c434a4785`: no_open, s90=0, o90=0, c90=0, stays=0, spend_rank=1.000 |
| 1 | Kontakty bez pobytové historie | 43,831 | eng=fading, open90=0.091, click90=0.000, sents90=1.24, last_open_med=289, stays=0.19, spend_rank=0.679 | `a0329a496045`: no_open, s90=1, o90=0, c90=0, stays=0, spend_rank=0.577<br>`c64494440618`: no_open, s90=1, o90=0, c90=0, stays=0, spend_rank=0.577 |
| 2 | Tichí příjemci | 60,946 | eng=fading, open90=0.013, click90=0.000, sents90=0.07, last_open_med=452, stays=1.17, spend_rank=0.749 | `bbe32d93a804`: no_open, s90=0, o90=0, c90=0, stays=1, spend_rank=0.800<br>`445175cc46c5`: no_open, s90=0, o90=0, c90=0, stays=1, spend_rank=0.801 |
| 3 | Hodnotní hosté | 4,449 | eng=healthy, open90=0.515, click90=0.365, sents90=2.09, last_open_med=56, stays=6.54, spend_rank=0.893 | `d0bc48e67382`: healthy, s90=1, o90=1, c90=0, stays=1, spend_rank=0.961<br>`7c4881ed5da2`: healthy, s90=1, o90=1, c90=0, stays=1, spend_rank=0.953 |
| 4 | Hodnotní hosté | 107,172 | eng=healthy, open90=0.520, click90=0.086, sents90=2.44, last_open_med=56, stays=1.17, spend_rank=0.873 | `32e3dfcc4b4e`: healthy, s90=2, o90=1, c90=0, stays=1, spend_rank=0.954<br>`b40def33a1b6`: healthy, s90=2, o90=1, c90=0, stays=1, spend_rank=0.954 |

Read: useful as a probabilistic alternative, but the current feature set makes two booking-value clusters and two silent clusters. It may be better after stronger feature normalization.

In [ ]:
# Inspect full generated JSON, including all examples and model paths.
approach_rows = []
for approach in report['approaches']:
    for cluster in approach['clusters']:
        profile = cluster['profile']
        approach_rows.append({
            'approach': approach['display_name'],
            'cluster_id': cluster['cluster_id'],
            'static_name': cluster['label'],
            'description': cluster.get('description'),
            'rows': profile['rows'],
            'engagement_mode': profile['engagement_state_mode'],
            'open_rate_90d_mean': profile['open_rate_90d_mean'],
            'click_rate_90d_mean': profile['click_rate_90d_mean'],
            'sents_90d_mean': profile['sents_90d_mean'],
            'days_since_last_open_median': profile['days_since_last_open_median'],
            'stays_total_mean': profile['stays_total_mean'],
            'rank_spend_total_mean': profile['rank_spend_total_mean'],
        })

pd.DataFrame(approach_rows)

In [ ]:
# Inspect anonymized guest examples for a selected approach/cluster.
SELECTED_APPROACH = 'cdp_kmeans'
SELECTED_CLUSTER = 3

for approach in report['approaches']:
    if approach['name'] == SELECTED_APPROACH:
        for cluster in approach['clusters']:
            if cluster['cluster_id'] == SELECTED_CLUSTER:
                display(pd.DataFrame(cluster['examples']))

## Recommendation

Use **Approach 1: CDP mixed k-means** as the first product candidate.

Why:

- It uses the widest CDP-style signal set: engagement, booking, profile, categorical context, and tenant-normalized ranks.
- Cluster sizes are more balanced than RFM BIRCH.
- It is simpler and easier to explain than latent SVD + Gaussian mixture.
- It can be serialized as a deterministic sklearn pipeline and called with feature parameters to return a cluster id.

Inference contract for the first service version:

1. If `reachable = 0`, return `NOT_TARGETABLE`.
2. If the contact is reachable but `sents_total = 0` or `engagement_state = never_sent`, return `COLD_CONTACTS`.
3. Otherwise run the learned model and return `cdp_kmeans:<cluster_id>`.

Do not use LLM in the model path. Human-readable names should remain a static dictionary outside inference.

## Detailnější CDP segmentation: k=8 vs k=10

Po prvním k=5 smoke testu dává větší počet clusterů smysl. Pro CDP segmentaci je obvykle praktičtější **8-10 segmentů**, pokud každý segment odpovídá jiné kampani nebo jiné prioritě cílení.

V aktuálních datech vychází lépe **k=10** než k=8, protože vytáhne samostatně:

- aktivní čtenáře,
- aktivní klikače,
- častější / hodnotnější hosty,
- různé stupně dřívějších hostů s klesající nebo nulovou odezvou,
- kontakty bez pobytové historie.

### Co v exportu chybí pro ještě lepší CDP segmentaci

Aktuální export **neobsahuje extra services, loyalty, e-commerce, check-in ani questionary signály**. Ve schématu jsou jen `tags_*`, `groups_*`, `segments_count`, `client_custom_*` a booking/mail engagement signály.

Pro segment typu “hosté co kupují extra services” je potřeba doplnit minimálně:

- `extra_services_orders_total`
- `extra_services_spend_total`
- `extra_services_orders_365d`
- `days_since_last_extra_service_order`
- `extra_services_categories_count`
- `extra_services_top_category`

### Doporučená k=10 taxonomie pro další iteraci

| Cluster | Navržený název | Proč je užitečný |
|---:|---|---|
| 0 | Neaktivní příjemci bez pobytové hodnoty | Nízká nebo žádná reakce, prakticky žádná booking hodnota. Nízká priorita. |
| 1 | Dřívější hosté s nízkou reakcí na maily | Mají pobytovou historii, ale slabý e-mail engagement. Win-back segment. |
| 2 | Oslovení bez reakce | Dostávají e-maily častěji, ale neotevírají. Testovat jiný kanál / hygienu kontaktů. |
| 3 | Častější hosté bez aktuální e-mailové reakce | Bookingově zajímavější hosté, ale mailingově chladní. Vysoká priorita reaktivace. |
| 4 | Aktivní čtenáři | Nedávno otevírají, méně klikají. Vhodné pro obsahové kampaně. |
| 5 | Dřívější hosté s vyhasínající odezvou | Historicky reagovali, dnes odezva slábne. Vhodné pro jemnou reaktivaci. |
| 6 | Newsletter kontakty se starší historickou odezvou | Malá pobytová hodnota, ale nějaká historická email odezva. Nižší priorita. |
| 7 | Dávní hosté bez aktuální odezvy | Starý engagement, nějaká pobytová historie. Spíš dlouhodobý win-back. |
| 8 | Často jezdící aktivní klikači | Nejzajímavější CDP segment: pobyty + recent open/click aktivita. Vhodné pro upsell/extra services. |
| 9 | Hodnotnější dřívější hosté bez odezvy | Vyšší booking/spend rank, ale slabá odezva. Personalizovaný win-back. |

Závěr: primární CDP k-means kandidát je teď **k=10** s ručně kontrolovaným statickým číselníkem názvů. LLM není potřeba v inference cestě.

In [ ]:
# Load k=8 / k=10 CDP k-means sweep results.
SWEEP_PATH = Path('../data/processed/clustering_experiments/cdp_k_sweep.json')
if not SWEEP_PATH.exists():
    SWEEP_PATH = Path('data/processed/clustering_experiments/cdp_k_sweep.json')
sweep = json.loads(SWEEP_PATH.read_text())

sweep_rows = []
for approach in sweep['approaches']:
    k = len(approach['clusters'])
    for cluster in approach['clusters']:
        profile = cluster['profile']
        sweep_rows.append({
            'k': k,
            'cluster_id': cluster['cluster_id'],
            'generated_name': cluster['label'],
            'rows': profile['rows'],
            'engagement_mode': profile['engagement_state_mode'],
            'open_rate_90d_mean': profile['open_rate_90d_mean'],
            'click_rate_90d_mean': profile['click_rate_90d_mean'],
            'sents_90d_mean': profile['sents_90d_mean'],
            'days_since_last_open_median': profile['days_since_last_open_median'],
            'stays_total_mean': profile['stays_total_mean'],
            'rank_spend_total_mean': profile['rank_spend_total_mean'],
        })

pd.DataFrame(sweep_rows).sort_values(['k', 'cluster_id'])